# 01 · Data Preparation — Enterprise PII Masker & Ticket Categorizer

**Goal:** turn the open-source [`ai4privacy/pii-masking-200k`](https://huggingface.co/datasets/ai4privacy/pii-masking-200k) corpus into a clean, balanced, strictly-JSON instruction-tuning dataset (ChatML `messages`) for QLoRA fine-tuning of a 3B SLM.

| Stage | What happens |
|---|---|
| 1. Ingest | Load the English split of `ai4privacy/pii-masking-200k` (character-level PII spans). |
| 2. Span repair & mapping | Validate every span offset, repair off-by-N offsets, map ~60 source labels to 7 enterprise tags. |
| 3. Quality gates | Drop rows with *out-of-scope secrets* (passwords, IBANs, crypto wallets…) that would teach the model to leak, length/charset filters, de-duplication. |
| 4. Weak supervision | Deterministic, auditable lexicon labeler for `ticket_category` + `urgency_level`, with an ambiguity margin filter. |
| 5. Synthetic augmentation | Rule-based (no paid LLM) enterprise tickets to cover `[API_KEY]`, `[IP_ADDRESS]`, `High` urgency and every department. |
| 6. Format + validate | Pydantic-validated JSON targets, leak checks, token-length checks with the real Qwen2.5 chat template. |
| 7. Split + publish | Stratified train/val/test → Parquet + Hugging Face Hub (dataset + shared prompt config). |

Runs on a **CPU Colab runtime** (no GPU needed for this notebook). Cost: **$0**.

## 0. Install dependencies
Colab already ships `pandas`, `numpy`, `scikit-learn`, `pydantic` v2 and `transformers`; we only add `datasets`.

In [ ]:
%pip install -q "datasets>=3.2.0,<5.0.0" "huggingface_hub>=0.27.0" "pydantic>=2.7,<3"

## 1. Configuration
Every tunable knob lives here. Secrets are read from **Colab Secrets** (`HF_TOKEN`) or the `HF_TOKEN` environment variable — never hard-coded.

In [ ]:
import ast
import hashlib
import json
import os
import random
import re
import string
import unicodedata
from collections import Counter
from pathlib import Path
from typing import Dict, List, Literal, Optional, Tuple

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ---- Source data ----
SOURCE_DATASET = "ai4privacy/pii-masking-200k"
SOURCE_LANGUAGE = "en"

# ---- Target dataset size & composition ----
TARGET_TOTAL = 4000            # final number of examples (within the 3k–5k brief)
SYNTHETIC_FRACTION = 0.20      # share of rule-based enterprise tickets (API keys, IPs, High urgency)
NEGATIVE_FRACTION = 0.08       # share of real rows with *no* in-scope PII (teaches "do nothing" behaviour)
MIN_CHARS, MAX_CHARS = 60, 1500
MIN_CATEGORY_SCORE = 3         # lexicon score needed to assign a non-General department
CATEGORY_MARGIN = 2            # top-1 must beat top-2 by this much, otherwise the row is ambiguous and dropped
MAX_CATEGORY_SHARE = 0.35      # no department may exceed this share of the real-row sample (stops General_Inquiry swamping)

# ---- Splits ----
VAL_SIZE = 300
TEST_SIZE = 300                # notebook 03 benchmarks on 200 of these

# ---- Tokenizer check (must match notebook 02) ----
BASE_TOKENIZER = "unsloth/Qwen2.5-3B-Instruct"
MAX_SEQ_LENGTH = 2048

# ---- Outputs ----
OUTPUT_DIR = Path("data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATASET_NAME = "enterprise-pii-ticket-sft"
PUSH_TO_HUB = True             # set False to keep everything local (Parquet only)
HF_PRIVATE = False             # public dataset = portfolio-visible

print(f"Output dir: {OUTPUT_DIR.resolve()}")

In [ ]:
def get_hf_token() -> Optional[str]:
    """Read HF token from Colab Secrets first, then from the environment."""
    try:
        from google.colab import userdata  # type: ignore
        token = userdata.get("HF_TOKEN")
        if token:
            return token
    except Exception:
        pass
    return os.environ.get("HF_TOKEN")


HF_TOKEN = get_hf_token()
HF_DATASET_REPO = None

if PUSH_TO_HUB and not HF_TOKEN:
    print("⚠️  No HF_TOKEN found (Colab: 🔑 Secrets → add HF_TOKEN with *write* scope). "
          "Continuing in local-only mode.")
    PUSH_TO_HUB = False

if PUSH_TO_HUB:
    from huggingface_hub import login, whoami
    login(token=HF_TOKEN, add_to_git_credential=False)
    HF_USERNAME = whoami(token=HF_TOKEN)["name"]
    HF_DATASET_REPO = f"{HF_USERNAME}/{DATASET_NAME}"
    print(f"✅ Authenticated as {HF_USERNAME}; dataset will be pushed to {HF_DATASET_REPO}")

## 2. Output contract (shared with training, evaluation and the FastAPI server)
The **system prompt**, **tag vocabulary** and **Pydantic schema** below are the single source of truth. They are exported to `prompt_config.json` and pushed alongside the dataset so notebooks 02/03 and `app/inference_server.py` load the *exact same* contract.

In [ ]:
from pydantic import BaseModel, ConfigDict, ValidationError, field_validator, model_validator

PII_TAGS: List[str] = ["PERSON", "EMAIL", "PHONE", "SSN", "CREDIT_CARD", "IP_ADDRESS", "API_KEY"]
CATEGORIES: List[str] = [
    "Billing_and_Payments",
    "Technical_Support",
    "Security_and_Access",
    "Legal_and_Compliance",
    "General_Inquiry",
]
URGENCY_LEVELS: List[str] = ["Low", "Medium", "High"]
TAG_PATTERN = re.compile(r"\[(" + "|".join(PII_TAGS) + r")\]")

PiiTag = Literal["PERSON", "EMAIL", "PHONE", "SSN", "CREDIT_CARD", "IP_ADDRESS", "API_KEY"]
Category = Literal[
    "Billing_and_Payments", "Technical_Support", "Security_and_Access",
    "Legal_and_Compliance", "General_Inquiry",
]
Urgency = Literal["Low", "Medium", "High"]


class TicketAnalysis(BaseModel):
    """Strict output schema. `extra='forbid'` rejects hallucinated keys."""
    model_config = ConfigDict(extra="forbid", strict=True)

    masked_text: str
    detected_pii_types: List[PiiTag]
    ticket_category: Category
    urgency_level: Urgency

    @field_validator("detected_pii_types")
    @classmethod
    def _unique_types(cls, v: List[str]) -> List[str]:
        if len(set(v)) != len(v):
            raise ValueError("detected_pii_types must not contain duplicates")
        return v

    @model_validator(mode="after")
    def _tags_match_types(self) -> "TicketAnalysis":
        tags_in_text = set(TAG_PATTERN.findall(self.masked_text))
        if tags_in_text != set(self.detected_pii_types):
            raise ValueError(
                f"Tags in masked_text {sorted(tags_in_text)} != detected_pii_types {sorted(self.detected_pii_types)}"
            )
        return self


SYSTEM_PROMPT = (
    "You are an enterprise data-protection and ticket-routing engine. "
    "Given a raw customer-support ticket or legal/HR snippet, return ONLY a single valid JSON object "
    "with exactly these keys, in this order:\n"
    '1. "masked_text": the input text reproduced exactly, except every piece of sensitive PII is replaced by '
    "one of these tags: [PERSON], [EMAIL], [PHONE], [SSN], [CREDIT_CARD], [IP_ADDRESS], [API_KEY]. "
    "A full name (first + last) becomes a single [PERSON]. Do not mask anything else and do not rephrase.\n"
    '2. "detected_pii_types": the list of tag names that appear in masked_text, without brackets, '
    "no duplicates, in this order: PERSON, EMAIL, PHONE, SSN, CREDIT_CARD, IP_ADDRESS, API_KEY. Use [] if none.\n"
    '3. "ticket_category": one of Billing_and_Payments, Technical_Support, Security_and_Access, '
    "Legal_and_Compliance, General_Inquiry.\n"
    '4. "urgency_level": one of Low, Medium, High.\n'
    "Output no markdown, no code fences and no explanation."
)
USER_TEMPLATE = "Analyze the following ticket.\n\n<ticket>\n{ticket}\n</ticket>"

PROMPT_CONFIG = {
    "version": "1.0.0",
    "system_prompt": SYSTEM_PROMPT,
    "user_template": USER_TEMPLATE,
    "pii_tags": PII_TAGS,
    "categories": CATEGORIES,
    "urgency_levels": URGENCY_LEVELS,
    "base_tokenizer": BASE_TOKENIZER,
    "max_seq_length": MAX_SEQ_LENGTH,
}
print(SYSTEM_PROMPT)

## 3. Ingest the open-source PII corpus
`ai4privacy/pii-masking-200k` (~209k rows, 4 languages) stores **character-offset spans** in `privacy_mask` (`[{value, start, end, label}, …]`). We keep English only.

In [ ]:
from datasets import load_dataset

raw = load_dataset(SOURCE_DATASET, split="train", token=HF_TOKEN)
print(raw)
print("Columns:", raw.column_names)

required_cols = {"source_text", "privacy_mask"}
missing = required_cols - set(raw.column_names)
if missing:
    raise RuntimeError(f"Source dataset schema changed — missing columns: {missing}")

keep_cols = ["source_text", "privacy_mask"] + (["language"] if "language" in raw.column_names else [])
raw_df = raw.select_columns(keep_cols).to_pandas()
if "language" in raw_df.columns:
    raw_df = raw_df[raw_df["language"] == SOURCE_LANGUAGE].reset_index(drop=True)
print(f"English rows: {len(raw_df):,}")

## 4. Span parsing, repair and label mapping
* Source labels are normalised (`FIRSTNAME_1` → `FIRSTNAME`).
* Every span is checked against the text; off-by-N offsets are **repaired** by locating the value near the stated offset, otherwise the row is rejected.
* `FIRSTNAME`/`MIDDLENAME`/`LASTNAME` spans separated only by spaces collapse into one `[PERSON]` — matching how a compliance officer would redact.
* Rows containing **out-of-scope secrets** (passwords, PINs, IBANs, wallets, CVVs…) are dropped: keeping them unmasked would literally train the model to leak secrets.

In [ ]:
LABEL_MAP: Dict[str, str] = {
    # people
    "FIRSTNAME": "PERSON", "MIDDLENAME": "PERSON", "LASTNAME": "PERSON", "FULLNAME": "PERSON",
    "NAME": "PERSON", "GIVENNAME": "PERSON", "SURNAME": "PERSON",
    # contact
    "EMAIL": "EMAIL",
    "PHONENUMBER": "PHONE", "PHONE": "PHONE", "TELEPHONENUM": "PHONE", "PHONEIMEI": None,
    # government id
    "SSN": "SSN", "SOCIALNUM": "SSN",
    # payment
    "CREDITCARDNUMBER": "CREDIT_CARD", "CREDITCARD": "CREDIT_CARD",
    # network
    "IP": "IP_ADDRESS", "IPV4": "IP_ADDRESS", "IPV6": "IP_ADDRESS",
    # identity mapping so synthetic rows can use the target tags directly
    **{t: t for t in PII_TAGS},
}
LABEL_MAP = {k: v for k, v in LABEL_MAP.items() if v is not None}

# Sensitive values we have no tag for. Rows containing them are excluded to avoid
# teaching the model that secrets may be left in clear text.
OUT_OF_SCOPE_SENSITIVE = {
    "PASSWORD", "PIN", "IBAN", "BIC", "ACCOUNTNUMBER", "CREDITCARDCVV", "MASKEDNUMBER",
    "BITCOINADDRESS", "ETHEREUMADDRESS", "LITECOINADDRESS", "PHONEIMEI", "VEHICLEVIN",
    "VEHICLEVRM", "DRIVERLICENSE", "PASSPORT", "IDCARD", "TAXNUM", "SSN/CCCD",
}


KNOWN_LABELS = set(LABEL_MAP) | OUT_OF_SCOPE_SENSITIVE


def normalise_label(label: str) -> str:
    """'FIRSTNAME_1' -> 'FIRSTNAME', 'EMAIL2' -> 'EMAIL', but 'IPV4'/'IPV6' stay intact."""
    label = str(label).upper().strip().strip("[]")
    label = re.sub(r"_\d+$", "", label)
    if label not in KNOWN_LABELS:
        stripped = re.sub(r"\d+$", "", label)
        if stripped in KNOWN_LABELS:
            label = stripped
    return label


assert normalise_label("IPV4") == "IPV4" and normalise_label("FIRSTNAME_1") == "FIRSTNAME"
assert normalise_label("[EMAIL_2]") == "EMAIL" and normalise_label("lastname2") == "LASTNAME"


def parse_privacy_mask(raw_mask) -> List[dict]:
    """Accepts list-of-dicts, numpy arrays of dicts, or JSON / Python-literal strings."""
    if raw_mask is None:
        return []
    if isinstance(raw_mask, str):
        s = raw_mask.strip()
        if not s:
            return []
        try:
            raw_mask = json.loads(s)
        except json.JSONDecodeError:
            raw_mask = ast.literal_eval(s)
    if isinstance(raw_mask, dict):
        raw_mask = [raw_mask]
    spans = []
    for item in list(raw_mask):
        spans.append({
            "label": normalise_label(item["label"]),
            "start": int(item["start"]),
            "end": int(item["end"]),
            "value": None if item.get("value") is None else str(item["value"]),
        })
    return sorted(spans, key=lambda s: (s["start"], s["end"]))


def repair_span(text: str, span: dict, window: int = 40) -> Optional[dict]:
    """Return a span whose offsets exactly cover its value, or None if unrecoverable."""
    start, end, value = span["start"], span["end"], span["value"]
    if value is None:
        if 0 <= start < end <= len(text):
            return {**span, "value": text[start:end]}
        return None
    if text[start:end] == value:
        return span
    lo = max(0, start - window)
    hi = min(len(text), end + window + len(value))
    idx = text.find(value, lo, hi)
    if idx == -1:
        occurrences = [m.start() for m in re.finditer(re.escape(value), text)]
        if len(occurrences) != 1:
            return None
        idx = occurrences[0]
    return {**span, "start": idx, "end": idx + len(value)}


PERSON_JOIN_GAP = re.compile(r"^[ \t]{1,2}$")


def mask_text(text: str, spans: List[dict]) -> Tuple[str, List[str], List[dict]]:
    """Replace in-scope spans with [TAG]. Returns (masked_text, ordered_types, applied_spans)."""
    in_scope = [{**s, "tag": LABEL_MAP[s["label"]]} for s in spans if s["label"] in LABEL_MAP]
    in_scope.sort(key=lambda s: (s["start"], -(s["end"] - s["start"])))

    applied: List[dict] = []
    for s in in_scope:
        if applied and s["start"] < applied[-1]["end"]:          # overlapping span → keep the earlier/longer one
            continue
        if (
            applied
            and s["tag"] == "PERSON"
            and applied[-1]["tag"] == "PERSON"
            and PERSON_JOIN_GAP.match(text[applied[-1]["end"]:s["start"]] or "")
        ):
            applied[-1] = {**applied[-1], "end": s["end"], "value": text[applied[-1]["start"]:s["end"]]}
            continue
        applied.append(dict(s))

    pieces, cursor = [], 0
    for s in applied:
        pieces.append(text[cursor:s["start"]])
        pieces.append(f"[{s['tag']}]")
        cursor = s["end"]
    pieces.append(text[cursor:])
    masked = "".join(pieces)

    found = {s["tag"] for s in applied}
    ordered_types = [t for t in PII_TAGS if t in found]
    return masked, ordered_types, applied


# Quick self-test on a hand-written example
_demo_text = "Hi, I am John  Smith (john.smith@acme.com). My SSN is 123-45-6789."
_demo_spans = [
    {"label": "FIRSTNAME_1", "start": 9, "end": 13, "value": "John"},
    {"label": "LASTNAME", "start": 15, "end": 20, "value": "Smith"},
    {"label": "EMAIL", "start": 23, "end": 42, "value": "john.smith@acme.com"},   # offset deliberately off by 1
    {"label": "SSN", "start": 54, "end": 65, "value": "123-45-6789"},
]
_fixed = [repair_span(_demo_text, {**s, "label": normalise_label(s["label"])}) for s in _demo_spans]
print(mask_text(_demo_text, _fixed)[:2])

## 5. Text quality gates

In [ ]:
CONTROL_CHARS = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")


def text_quality_ok(text: str) -> bool:
    if not isinstance(text, str):
        return False
    t = text.strip()
    if not (MIN_CHARS <= len(t) <= MAX_CHARS):
        return False
    if CONTROL_CHARS.search(t):
        return False
    if re.search(r"\[[A-Z_]{3,}\]", t):
        return False                                       # already-masked text would poison targets
    letters = sum(ch.isalpha() for ch in t)
    if letters / max(len(t), 1) < 0.55:
        return False
    non_ascii = sum(ord(ch) > 127 for ch in t)
    if non_ascii / max(len(t), 1) > 0.05:
        return False
    if len(t.split()) < 8:
        return False
    return True


def normalise_for_dedup(text: str) -> str:
    t = unicodedata.normalize("NFKC", text).lower()
    t = re.sub(r"\d", "0", t)
    t = re.sub(r"\s+", " ", t).strip()
    return hashlib.md5(t.encode("utf-8")).hexdigest()

## 6. Weak supervision: department & urgency labels
The PII corpus has no routing labels, so we derive them with a **transparent lexicon labeler** (weighted, word-boundary regex). Two safeguards keep label noise low:
1. A row is only assigned a department if its score ≥ `MIN_CATEGORY_SCORE`; otherwise it's `General_Inquiry`.
2. If the top-2 department scores are within `CATEGORY_MARGIN`, the row is **ambiguous and dropped**.

This is documented as a limitation in the README; the synthetic set (Section 7) provides gold-labelled routing examples.

In [ ]:
CATEGORY_LEXICON: Dict[str, Dict[str, int]] = {
    "Billing_and_Payments": {
        "invoice": 3, "invoices": 3, "billing": 3, "billed": 3, "payment": 3, "payments": 3, "refund": 3,
        "charged": 3, "chargeback": 3, "overcharged": 3, "credit card": 2, "subscription": 2, "pricing": 2,
        "overdue": 2, "receipt": 2, "transaction": 2, "transactions": 2, "payroll": 2, "reimbursement": 3,
        "balance": 2, "fee": 1, "fees": 1, "purchase": 1, "bank transfer": 2, "wire transfer": 2, "debit": 2,
        "account statement": 2, "tax": 1, "price": 1, "amount": 1, "pay": 1, "paid": 1,
    },
    "Technical_Support": {
        "error": 3, "errors": 3, "bug": 3, "crash": 3, "crashes": 3, "crashed": 3, "not working": 3,
        "outage": 3, "troubleshoot": 3, "troubleshooting": 3, "server": 2, "servers": 2, "install": 2,
        "installation": 2, "software": 2, "network": 2, "latency": 2, "timeout": 2, "configuration": 2,
        "database": 2, "deployment": 2, "upgrade": 2, "reboot": 2, "connectivity": 2, "browser": 2,
        "laptop": 2, "printer": 2, "vpn": 2, "firmware": 2, "app": 1, "device": 1, "system": 1,
        "website": 1, "technical": 2, "it support": 3, "update": 1,
    },
    "Security_and_Access": {
        "password": 3, "unauthorized": 3, "breach": 3, "compromised": 3, "phishing": 3, "credentials": 3,
        "api key": 3, "2fa": 3, "mfa": 3, "two-factor": 3, "locked out": 3, "suspicious": 3, "malware": 3,
        "account takeover": 3, "security": 2, "access": 2, "permission": 2, "permissions": 2, "token": 2,
        "revoke": 2, "login": 2, "log in": 2, "authentication": 2, "encrypted": 1, "encryption": 1,
        "leak": 2, "leaked": 3, "firewall": 2, "intrusion": 3, "hacked": 3, "verification code": 2,
    },
    "Legal_and_Compliance": {
        "gdpr": 3, "compliance": 3, "legal": 3, "lawsuit": 3, "court": 3, "attorney": 3, "lawyer": 3,
        "subpoena": 3, "hipaa": 3, "nda": 3, "litigation": 3, "harassment": 3, "grievance": 3,
        "contract": 2, "agreement": 2, "regulation": 2, "regulatory": 2, "audit": 2, "consent": 2,
        "liability": 2, "clause": 2, "dispute": 2, "termination": 2, "disciplinary": 2, "tribunal": 3,
        "intellectual property": 3, "copyright": 2, "policy": 1, "privacy": 1, "terms": 1, "employment": 1,
        "jurisdiction": 3, "settlement": 2, "legislation": 2,
    },
}
CATEGORY_PRIORITY = ["Security_and_Access", "Legal_and_Compliance", "Billing_and_Payments", "Technical_Support"]


def _compile_lexicon(lex: Dict[str, int]) -> List[Tuple[re.Pattern, int]]:
    return [(re.compile(r"(?<![a-z0-9])" + re.escape(k) + r"(?![a-z0-9])"), w) for k, w in lex.items()]


COMPILED_LEXICON = {cat: _compile_lexicon(lex) for cat, lex in CATEGORY_LEXICON.items()}

URGENCY_HIGH = [
    "urgent", "urgently", "immediately", "asap", "emergency", "critical", "breach", "compromised", "fraud",
    "fraudulent", "unauthorized", "lawsuit", "subpoena", "outage", "data leak", "as soon as possible",
    "within 24 hours", "right away", "escalate", "escalation", "production down", "system down", "hacked",
    "legal action", "final notice",
]
URGENCY_MEDIUM = [
    "soon", "delay", "delayed", "issue", "problem", "error", "failed", "failure", "overdue", "follow up",
    "follow-up", "reminder", "pending", "deadline", "not working", "complaint", "incorrect", "disputed",
    "this week", "by friday", "unable to",
]
URGENCY_HIGH_RE = _compile_lexicon({k: 1 for k in URGENCY_HIGH})
URGENCY_MEDIUM_RE = _compile_lexicon({k: 1 for k in URGENCY_MEDIUM})


def score_categories(text: str) -> Dict[str, int]:
    t = text.lower()
    return {cat: sum(w * len(p.findall(t)) for p, w in pats) for cat, pats in COMPILED_LEXICON.items()}


def classify_category(text: str) -> Optional[str]:
    """Returns a category, or None when the row is ambiguous and should be dropped."""
    scores = score_categories(text)
    ranked = sorted(scores.items(), key=lambda kv: (-kv[1], CATEGORY_PRIORITY.index(kv[0])))
    (top_cat, top), (_, second) = ranked[0], ranked[1]
    if top < MIN_CATEGORY_SCORE:
        return "General_Inquiry" if top <= 1 else None      # weak single hits are ambiguous
    if top - second < CATEGORY_MARGIN:
        return None
    return top_cat


def classify_urgency(text: str, category: str, pii_types: List[str]) -> str:
    t = text.lower()
    high = sum(len(p.findall(t)) for p, _ in URGENCY_HIGH_RE)
    medium = sum(len(p.findall(t)) for p, _ in URGENCY_MEDIUM_RE)
    if high >= 1:
        level = "High"
    elif medium >= 1:
        level = "Medium"
    else:
        level = "Low"
    # Business rule: exposed secrets/financial identifiers in a security ticket are never Low.
    if category == "Security_and_Access" and {"SSN", "CREDIT_CARD", "API_KEY"} & set(pii_types) and level == "Low":
        level = "Medium"
    return level


for _t in [
    "Hi, I was charged twice on my last invoice and need a refund.",
    "Our VPN keeps crashing with a timeout error after the firmware upgrade.",
    "Urgent: we detected unauthorized login attempts and suspect a breach.",
    "Please review the NDA clause before the GDPR audit.",
    "Can you tell me your office opening hours?",
]:
    _c = classify_category(_t)
    print(f"{_c!s:22} | {classify_urgency(_t, _c or 'General_Inquiry', [])}")

## 7. Process the real corpus

In [ ]:
def process_real_row(text: str, raw_mask) -> Tuple[Optional[dict], str]:
    """Returns (record, reject_reason). record is None if rejected."""
    if not text_quality_ok(text):
        return None, "quality"
    try:
        spans = parse_privacy_mask(raw_mask)
    except Exception:
        return None, "unparseable_mask"
    if any(s["label"] in OUT_OF_SCOPE_SENSITIVE for s in spans):
        return None, "out_of_scope_secret"
    repaired = []
    for s in spans:
        r = repair_span(text, s)
        if r is None:
            if s["label"] in LABEL_MAP:
                return None, "bad_span"
            continue                                       # misaligned *irrelevant* span: harmless
        repaired.append(r)
    masked, types, applied = mask_text(text, repaired)
    leaked = [s["value"] for s in applied if s["value"] and len(s["value"]) >= 4 and s["value"] in masked]
    if leaked:
        return None, "leak_after_mask"
    category = classify_category(text)
    if category is None:
        return None, "ambiguous_category"
    urgency = classify_urgency(text, category, types)
    return {
        "source_text": text,
        "masked_text": masked,
        "detected_pii_types": types,
        "ticket_category": category,
        "urgency_level": urgency,
        "origin": "ai4privacy",
    }, "ok"


records, reasons = [], Counter()
for text, mask in zip(raw_df["source_text"].tolist(), raw_df["privacy_mask"].tolist()):
    rec, why = process_real_row(text, mask)
    reasons[why] += 1
    if rec is not None:
        records.append(rec)

real_df = pd.DataFrame(records)
real_df["dedup_key"] = real_df["source_text"].map(normalise_for_dedup)
before = len(real_df)
real_df = real_df.drop_duplicates("dedup_key").reset_index(drop=True)
reasons["duplicate"] = before - len(real_df)

print("Filter funnel:")
for k, v in reasons.most_common():
    print(f"  {k:22} {v:>7,}")
print(f"\nCandidate pool: {len(real_df):,}")
print(real_df["ticket_category"].value_counts())

## 8. Rule-based synthetic enterprise tickets ($0, no LLM APIs)
The public corpus has **no API keys**, few IPs and almost no explicitly urgent tickets. We generate realistic, noisy tickets from hand-written templates with gold labels.

* Credit cards are **Luhn-valid test-range** numbers; phone numbers use the reserved **555** exchange; e-mail domains are fictitious.
* API keys use generic prefixes (`api_`, `svc_`, `tok_`, `key-`, `ent_`) — deliberately *not* real provider formats, so GitHub/HF secret scanners don't flag the dataset.

In [ ]:
rng = random.Random(SEED)

FIRST_NAMES = [
    "James", "Maria", "Ahmed", "Priya", "Chen", "Olivia", "Mateo", "Fatima", "Liam", "Aisha", "Noah", "Sofia",
    "Hiroshi", "Elena", "Kwame", "Zara", "Lucas", "Mei", "Omar", "Hannah", "Diego", "Ingrid", "Ravi", "Chloe",
    "Tariq", "Grace", "Mikhail", "Amara", "Daniel", "Yuki", "Bilal", "Emma", "Santiago", "Nadia", "Ethan", "Leila",
]
LAST_NAMES = [
    "Anderson", "Garcia", "Khan", "Sharma", "Wei", "Thompson", "Rossi", "Haddad", "O'Connor", "Okafor", "Müller",
    "Fernandes", "Tanaka", "Petrova", "Mensah", "Hussain", "Silva", "Nguyen", "Kowalski", "Johansson", "Patel",
    "Dubois", "Ali", "Brennan", "Yilmaz", "Kim", "Novak", "Mahmood", "Reyes", "Lindqvist",
]
EMAIL_DOMAINS = ["northwind-corp.com", "contoso.io", "globex-intl.net", "initech.co", "umbrella-health.org",
                 "stark-logistics.com", "acmefinance.biz", "wayne-enterprises.net"]


def gen_person() -> str:
    first, last = rng.choice(FIRST_NAMES), rng.choice(LAST_NAMES)
    return f"{first} {last}" if rng.random() < 0.75 else first


def gen_email(person: Optional[str] = None) -> str:
    base = (person or f"{rng.choice(FIRST_NAMES)} {rng.choice(LAST_NAMES)}").lower()
    base = re.sub(r"[^a-z ]", "", unicodedata.normalize("NFKD", base).encode("ascii", "ignore").decode())
    parts = base.split() or ["user"]
    local = rng.choice([".".join(parts), parts[0] + str(rng.randint(1, 99)), parts[0][0] + parts[-1]])
    return f"{local}@{rng.choice(EMAIL_DOMAINS)}"


def gen_phone() -> str:
    area = rng.randint(201, 989)
    line = rng.randint(100, 199)                           # 555-0100..0199 is reserved for fiction
    return rng.choice([
        f"({area}) 555-0{line}", f"+1 {area}-555-0{line}", f"{area}.555.0{line}", f"+1 ({area}) 555 0{line}",
    ])


def gen_ssn() -> str:
    area = rng.choice([rng.randint(100, 665), rng.randint(667, 899)])
    return f"{area:03d}-{rng.randint(1, 99):02d}-{rng.randint(1, 9999):04d}"


def _luhn_complete(prefix: str, length: int) -> str:
    digits = [int(d) for d in prefix] + [rng.randint(0, 9) for _ in range(length - len(prefix) - 1)]
    total = 0
    for i, d in enumerate(reversed(digits)):
        if i % 2 == 0:
            d *= 2
            if d > 9:
                d -= 9
        total += d
    check = (10 - total % 10) % 10
    return "".join(map(str, digits + [check]))


def luhn_valid(number: str) -> bool:
    digits = [int(d) for d in re.sub(r"\D", "", number)]
    total = 0
    for i, d in enumerate(reversed(digits)):
        if i % 2 == 1:
            d *= 2
            if d > 9:
                d -= 9
        total += d
    return total % 10 == 0


def gen_credit_card() -> str:
    number = _luhn_complete(rng.choice(["4111", "4000", "5555", "5105", "2223"]), 16)
    sep = rng.choice([" ", "-", ""])
    return sep.join(number[i:i + 4] for i in range(0, 16, 4))


def gen_ip() -> str:
    if rng.random() < 0.15:
        return ":".join(f"{rng.randint(0, 65535):x}" for _ in range(8))
    first = rng.choice([23, 34, 52, 64, 81, 98, 104, 142, 172, 185, 203, 10, 192])
    return f"{first}.{rng.randint(0, 255)}.{rng.randint(0, 255)}.{rng.randint(1, 254)}"


def gen_api_key() -> str:
    alphabet = string.ascii_letters + string.digits
    body = "".join(rng.choice(alphabet) for _ in range(rng.choice([24, 32, 40])))
    hexbody = "".join(rng.choice("0123456789abcdef") for _ in range(32))
    return rng.choice([f"api_{body}", f"svc_{body}", f"tok_{body}", f"key-{hexbody}", f"ent_{hexbody}"])


assert all(luhn_valid(gen_credit_card()) for _ in range(500)), "Luhn generator broken"

SLOT_GENERATORS = {
    "PERSON": gen_person, "PERSON2": gen_person, "EMAIL": gen_email, "PHONE": gen_phone, "SSN": gen_ssn,
    "CREDIT_CARD": gen_credit_card, "IP_ADDRESS": gen_ip, "IP_ADDRESS2": gen_ip, "API_KEY": gen_api_key,
}

# (category, urgency, template). Slots: {PERSON} {PERSON2} {EMAIL} {PHONE} {SSN} {CREDIT_CARD} {IP_ADDRESS} {IP_ADDRESS2} {API_KEY}
TEMPLATES: List[Tuple[str, str, str]] = [
    # ---- Security_and_Access ----
    ("Security_and_Access", "High",
     "URGENT - one of our devs ({PERSON}) accidentally committed our production key {API_KEY} to a public repo about an hour ago. Please revoke it immediately and confirm. Reach me at {PHONE}."),
    ("Security_and_Access", "High",
     "We are seeing hundreds of failed login attempts on the admin portal from {IP_ADDRESS} and {IP_ADDRESS2}. Looks like credential stuffing. Can security block these IPs right away? - {PERSON}"),
    ("Security_and_Access", "High",
     "hi, my account was hacked last night. someone changed the recovery email to {EMAIL} and I can't log in anymore. my name is {PERSON}, please lock the account asap"),
    ("Security_and_Access", "Medium",
     "Hello team, please rotate the integration token {API_KEY} used by the billing sync job. It is due for its 90-day rotation this week. Owner: {PERSON} ({EMAIL})."),
    ("Security_and_Access", "Medium",
     "I'm locked out after too many MFA attempts. Username is tied to {EMAIL}. Last successful login was from {IP_ADDRESS}. Can someone reset my 2FA? Thanks, {PERSON}"),
    ("Security_and_Access", "Low",
     "Could you grant {PERSON} read-only access to the analytics workspace? Their work email is {EMAIL}. No rush, sometime next week is fine."),
    ("Security_and_Access", "Low",
     "Requesting a new service account for the reporting pipeline. Please whitelist our egress IP {IP_ADDRESS}. Contact: {PERSON}."),
    ("Security_and_Access", "High",
     "Phishing alert: {PERSON} received an email asking to confirm their SSN {SSN} and replied before realising. Please advise on next steps immediately, this is a potential data breach."),
    # ---- Technical_Support ----
    ("Technical_Support", "High",
     "PRODUCTION DOWN. Our API gateway at {IP_ADDRESS} returns 502 for every request since 09:40 UTC. Header used: Authorization: Bearer {API_KEY}. Escalate to on-call please. {PERSON} {PHONE}"),
    ("Technical_Support", "Medium",
     "Hi support, the desktop app crashes every time I export a report to PDF. Error code 0x80070057. I'm on version 4.2.1. You can reach me at {EMAIL} or {PHONE}. - {PERSON}"),
    ("Technical_Support", "Medium",
     "Our webhook calls to your endpoint keep timing out (30s). Source server {IP_ADDRESS}. Can you check if we're being rate limited? Regards, {PERSON}"),
    ("Technical_Support", "Low",
     "Quick question: is there a way to change the default date format in the dashboard to DD/MM/YYYY? Not urgent. Thanks, {PERSON}"),
    ("Technical_Support", "Low",
     "Hello, I'd like help configuring SSO for our team of 12. Our IT contact is {PERSON}, {EMAIL}. Whenever you have availability works for us."),
    ("Technical_Support", "Medium",
     "the printer on floor 3 is not working again, shows paper jam but there is no paper stuck. asset tag PR-3391. call {PERSON} on {PHONE} if you need access to the room"),
    ("Technical_Support", "High",
     "Critical: database replication between {IP_ADDRESS} and {IP_ADDRESS2} stopped and customer orders are failing. We need an engineer immediately. Contact {PERSON} at {PHONE}."),
    # ---- Billing_and_Payments ----
    ("Billing_and_Payments", "High",
     "I see three charges I did not make on my card {CREDIT_CARD}. This is fraud, please block the card and refund immediately. {PERSON}, {PHONE}"),
    ("Billing_and_Payments", "Medium",
     "Hi, I was charged twice for the March invoice (INV-20931). Card ending used was {CREDIT_CARD}. Please refund the duplicate. Thanks, {PERSON} - {EMAIL}"),
    ("Billing_and_Payments", "Medium",
     "Our invoice is overdue because the payment portal keeps rejecting our corporate card. Can you send a payment link to {EMAIL}? Accounts payable: {PERSON}."),
    ("Billing_and_Payments", "Low",
     "Could you update the billing contact on our subscription to {PERSON} ({EMAIL}, {PHONE})? No other changes needed."),
    ("Billing_and_Payments", "Low",
     "Hello, can I get a copy of last year's receipts for tax purposes? Please send them to {EMAIL}. Best, {PERSON}"),
    ("Billing_and_Payments", "Medium",
     "Payroll issue: my reimbursement for travel expenses has been pending for 3 weeks. Employee {PERSON}, SSN on file {SSN}. Please follow up."),
    ("Billing_and_Payments", "Low",
     "Hi, we would like to switch from monthly to annual billing. The card on file is {CREDIT_CARD}. Please let {PERSON} know the new pricing."),
    # ---- Legal_and_Compliance ----
    ("Legal_and_Compliance", "High",
     "We received a subpoena requesting all records related to {PERSON} (SSN {SSN}). Legal needs these compiled within 24 hours. Contact counsel at {EMAIL}."),
    ("Legal_and_Compliance", "Medium",
     "GDPR data subject access request from {PERSON}, {EMAIL}. They are requesting all personal data we hold. Statutory deadline is in 30 days."),
    ("Legal_and_Compliance", "Medium",
     "HR grievance: {PERSON} has filed a formal harassment complaint against their manager {PERSON2}. Please open a confidential case and contact them at {PHONE}."),
    ("Legal_and_Compliance", "Low",
     "Please review the attached NDA for our new vendor before signing. The vendor contact is {PERSON} ({EMAIL}). Target signature date is end of month."),
    ("Legal_and_Compliance", "High",
     "Final notice: opposing counsel has threatened legal action over the contract dispute with {PERSON}. Need an attorney assigned immediately. Call {PHONE}."),
    ("Legal_and_Compliance", "Low",
     "For the annual compliance audit, can you confirm our data retention policy for employee records? Point of contact: {PERSON}, {EMAIL}."),
    ("Legal_and_Compliance", "Medium",
     "Employment termination paperwork for {PERSON} (SSN {SSN}) is missing the signed separation agreement. Please follow up with legal this week."),
    # ---- General_Inquiry ----
    ("General_Inquiry", "Low",
     "Hi there, what are your office hours during the holiday season? Thanks, {PERSON}"),
    ("General_Inquiry", "Low",
     "Hello! I'd like to know if you offer on-site training sessions for new teams. Please reach out to {EMAIL}."),
    ("General_Inquiry", "Low",
     "Can someone send me the brochure for your enterprise plan? My number is {PHONE}. Regards, {PERSON}"),
    ("General_Inquiry", "Medium",
     "Following up on my question from last week about partnership opportunities - still waiting on a reply. {PERSON}, {EMAIL}"),
    ("General_Inquiry", "Low",
     "Just wanted to say the onboarding session with {PERSON} was great. Who should I contact about scheduling another one?"),
    ("General_Inquiry", "Low",
     "Do you have a physical office in Dubai or Karachi? We are looking to visit in the spring."),
]

SIGNATURES = ["", "", "", "\n\nSent from my iPhone", "\n\n--\nThis email and any attachments are confidential.",
              "\n\nRegards", "\n\nThanks in advance!"]
GREETINGS = ["", "", "", "Hi team,\n", "Hello,\n", "Dear Support,\n"]

SLOT_RE = re.compile(r"\{(" + "|".join(SLOT_GENERATORS) + r")\}")


def render_template(template: str) -> Tuple[str, List[dict]]:
    """Fill slots left-to-right while recording exact character spans."""
    values: Dict[str, str] = {}
    person_for_email = None
    out, spans, cursor = [], [], 0
    for m in SLOT_RE.finditer(template):
        out.append(template[cursor:m.start()])
        slot = m.group(1)
        if slot not in values:
            if slot == "EMAIL":
                values[slot] = gen_email(person_for_email if rng.random() < 0.6 else None)
            else:
                values[slot] = SLOT_GENERATORS[slot]()
            if slot == "PERSON":
                person_for_email = values[slot]
        start = sum(len(p) for p in out)
        out.append(values[slot])
        tag = re.sub(r"\d+$", "", slot)
        spans.append({"label": tag, "start": start, "end": start + len(values[slot]), "value": values[slot]})
        cursor = m.end()
    out.append(template[cursor:])
    return "".join(out), spans


def add_noise(text: str, spans: List[dict]) -> Tuple[str, List[dict]]:
    """Prepend greeting / append signature (shifts spans), and randomly lower-case the first letter."""
    already_greets = re.match(r"(?i)^(hi|hello|dear|hey)\b", text) is not None
    greeting = "" if already_greets else rng.choice(GREETINGS)
    signature = rng.choice(SIGNATURES)
    shifted = [{**s, "start": s["start"] + len(greeting), "end": s["end"] + len(greeting)} for s in spans]
    text = greeting + text + signature
    first_char_is_pii = bool(shifted) and shifted[0]["start"] == 0
    if not first_char_is_pii and text[:1].isupper() and rng.random() < 0.15:
        text = text[0].lower() + text[1:]              # sloppy real-world casing; never alters a PII value
    return text, shifted


def generate_synthetic(n: int) -> pd.DataFrame:
    rows = []
    by_cat: Dict[str, List[Tuple[str, str, str]]] = {}
    for t in TEMPLATES:
        by_cat.setdefault(t[0], []).append(t)
    cats = sorted(by_cat)
    for i in range(n):
        cat = cats[i % len(cats)]                          # round-robin → balanced departments
        _, urgency, template = rng.choice(by_cat[cat])
        text, spans = render_template(template)
        text, spans = add_noise(text, spans)
        masked, types, _ = mask_text(text, spans)
        rows.append({
            "source_text": text, "masked_text": masked, "detected_pii_types": types,
            "ticket_category": cat, "urgency_level": urgency, "origin": "synthetic",
        })
    df = pd.DataFrame(rows)
    df["dedup_key"] = df["source_text"].map(normalise_for_dedup)
    return df


n_synth = int(round(TARGET_TOTAL * SYNTHETIC_FRACTION))
synth_df = generate_synthetic(int(n_synth * 1.3)).drop_duplicates("dedup_key")
synth_df = synth_df.sample(n=min(n_synth, len(synth_df)), random_state=SEED).reset_index(drop=True)
print(f"Synthetic rows: {len(synth_df):,}")
print(synth_df[["ticket_category", "urgency_level"]].value_counts().sort_index())
print("\nExample:\n", synth_df.iloc[0]["masked_text"])

## 9. Balanced sampling of the real corpus
Department-balanced sampling of positives (rows with ≥1 in-scope PII) plus a fixed share of negatives.

In [ ]:
def balanced_sample(df: pd.DataFrame, n_total: int, seed: int, max_share: float = 1.0) -> pd.DataFrame:
    """Sample up to n_total rows as evenly as possible across ticket_category, redistributing shortfall,
    with no category exceeding max_share of n_total."""
    if n_total <= 0 or df.empty:
        return df.head(0).copy()
    groups = {c: g.sample(frac=1.0, random_state=seed) for c, g in df.groupby("ticket_category")}
    cap = {c: min(len(g), max(1, int(np.ceil(max_share * n_total)))) for c, g in groups.items()}
    quota = {c: 0 for c in groups}
    remaining = n_total
    while remaining > 0:
        open_cats = [c for c in groups if quota[c] < cap[c]]
        if not open_cats:
            break
        share = max(1, remaining // len(open_cats))
        for c in open_cats:
            take = min(share, cap[c] - quota[c], remaining)
            quota[c] += take
            remaining -= take
            if remaining == 0:
                break
    return pd.concat([groups[c].head(quota[c]) for c in groups]).reset_index(drop=True)


n_real = TARGET_TOTAL - len(synth_df)
n_neg = int(round(n_real * NEGATIVE_FRACTION))
pos_pool = real_df[real_df["detected_pii_types"].map(len) > 0]
neg_pool = real_df[real_df["detected_pii_types"].map(len) == 0]
neg_sample = balanced_sample(neg_pool, min(n_neg, len(neg_pool)), SEED)
pos_sample = balanced_sample(pos_pool, n_real - len(neg_sample), SEED, max_share=MAX_CATEGORY_SHARE)

dataset_df = pd.concat([pos_sample, neg_sample, synth_df], ignore_index=True)
dataset_df = dataset_df.drop_duplicates("dedup_key").sample(frac=1.0, random_state=SEED).reset_index(drop=True)
print(f"Final pool: {len(dataset_df):,}  (real positives={len(pos_sample):,}, real negatives={len(neg_sample):,}, synthetic={len(synth_df):,})")
if len(dataset_df) < 3000:
    print("⚠️  Fewer than 3,000 rows survived filtering — consider lowering CATEGORY_MARGIN or raising MAX_CHARS.")

## 10. Build JSON targets + ChatML messages, then validate everything
Every target is serialised with a **fixed key order** and validated by the Pydantic schema. We also re-run a leak check (no original PII value may survive in `masked_text`) and measure token lengths with the real Qwen2.5 chat template.

In [ ]:
def build_target(row: pd.Series) -> dict:
    return {
        "masked_text": row["masked_text"],
        "detected_pii_types": list(row["detected_pii_types"]),
        "ticket_category": row["ticket_category"],
        "urgency_level": row["urgency_level"],
    }


def to_messages(source_text: str, target_json: str) -> List[dict]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_TEMPLATE.format(ticket=source_text)},
        {"role": "assistant", "content": target_json},
    ]


dataset_df["target"] = dataset_df.apply(build_target, axis=1)
dataset_df["target_json"] = dataset_df["target"].map(lambda d: json.dumps(d, ensure_ascii=False))
dataset_df["messages"] = [to_messages(s, t) for s, t in zip(dataset_df["source_text"], dataset_df["target_json"])]

schema_errors = []
for i, tj in enumerate(dataset_df["target_json"]):
    try:
        TicketAnalysis.model_validate_json(tj)
    except ValidationError as e:
        schema_errors.append((i, str(e)[:200]))
if schema_errors:
    raise AssertionError(f"{len(schema_errors)} targets fail the schema, first: {schema_errors[0]}")
print(f"✅ {len(dataset_df):,}/{len(dataset_df):,} targets pass the Pydantic schema")

# Masked text must equal the source outside of the tagged regions.
def unmasked_skeleton_matches(source: str, masked: str) -> bool:
    pattern = "^" + "".join(
        ".+?" if TAG_PATTERN.fullmatch(part) else re.escape(part)
        for part in re.split(r"(\[(?:" + "|".join(PII_TAGS) + r")\])", masked) if part
    ) + "$"
    return re.match(pattern, source, flags=re.DOTALL) is not None


bad_skeleton = [i for i, (s, m) in enumerate(zip(dataset_df["source_text"], dataset_df["masked_text"]))
                if not unmasked_skeleton_matches(s, m)]
if bad_skeleton:
    raise AssertionError(f"{len(bad_skeleton)} rows where masked_text is not a faithful masking of source_text")
print("✅ Every masked_text is a faithful, tag-only edit of its source_text")

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_TOKENIZER, token=HF_TOKEN)
def count_chat_tokens(messages: List[dict]) -> int:
    rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return len(tokenizer(rendered, add_special_tokens=False)["input_ids"])


dataset_df["n_tokens"] = [count_chat_tokens(m) for m in dataset_df["messages"]]
too_long = int((dataset_df["n_tokens"] > MAX_SEQ_LENGTH).sum())
dataset_df = dataset_df[dataset_df["n_tokens"] <= MAX_SEQ_LENGTH].reset_index(drop=True)
print(f"Token length — p50={dataset_df['n_tokens'].median():.0f}, p95={dataset_df['n_tokens'].quantile(0.95):.0f}, "
      f"max={dataset_df['n_tokens'].max()}  |  dropped {too_long} rows > {MAX_SEQ_LENGTH}")

## 11. Stratified train / validation / test split
Stratified on `ticket_category × urgency_level` (rare combinations fall back to `ticket_category`) so all three splits share the same label distribution.

In [ ]:
from sklearn.model_selection import train_test_split

strat = dataset_df["ticket_category"] + "|" + dataset_df["urgency_level"]
counts = strat.value_counts()
strat = strat.where(strat.map(counts) >= 10, dataset_df["ticket_category"])
dataset_df["strat_key"] = strat

train_val, test_df = train_test_split(dataset_df, test_size=TEST_SIZE, stratify=dataset_df["strat_key"], random_state=SEED)
train_df, val_df = train_test_split(train_val, test_size=VAL_SIZE, stratify=train_val["strat_key"], random_state=SEED)

splits = {"train": train_df, "validation": val_df, "test": test_df}
for name, df in splits.items():
    print(f"\n{name}: {len(df):,} rows")
    print((df["ticket_category"].value_counts(normalize=True) * 100).round(1).to_string())

assert not (set(train_df["dedup_key"]) & set(test_df["dedup_key"])), "train/test leakage"
assert not (set(train_df["dedup_key"]) & set(val_df["dedup_key"])), "train/val leakage"
print("\n✅ No duplicate leakage across splits")

## 12. Dataset report

In [ ]:
def pii_type_counts(df: pd.DataFrame) -> Dict[str, int]:
    c = Counter(t for types in df["detected_pii_types"] for t in types)
    return {t: c.get(t, 0) for t in PII_TAGS}


report = {
    "source_dataset": SOURCE_DATASET,
    "total_rows": int(len(dataset_df)),
    "split_sizes": {k: int(len(v)) for k, v in splits.items()},
    "origin": dataset_df["origin"].value_counts().to_dict(),
    "ticket_category": dataset_df["ticket_category"].value_counts().to_dict(),
    "urgency_level": dataset_df["urgency_level"].value_counts().to_dict(),
    "pii_type_counts": pii_type_counts(dataset_df),
    "rows_without_pii": int((dataset_df["detected_pii_types"].map(len) == 0).sum()),
    "token_length": {
        "p50": float(dataset_df["n_tokens"].median()),
        "p95": float(dataset_df["n_tokens"].quantile(0.95)),
        "max": int(dataset_df["n_tokens"].max()),
    },
    "filter_funnel": dict(reasons),
    "seed": SEED,
}
print(json.dumps(report, indent=2))

print("\n--- Sample training example (assistant target) ---")
print(train_df.iloc[0]["source_text"])
print(json.dumps(train_df.iloc[0]["target"], indent=2, ensure_ascii=False))

## 13. Persist: Parquet + JSONL + prompt config

In [ ]:
EXPORT_COLS = ["messages", "source_text", "target_json", "masked_text", "detected_pii_types",
               "ticket_category", "urgency_level", "origin", "n_tokens"]

for name, df in splits.items():
    out = df[EXPORT_COLS].reset_index(drop=True)
    out.to_parquet(OUTPUT_DIR / f"{name}.parquet", index=False)
    out.to_json(OUTPUT_DIR / f"{name}.jsonl", orient="records", lines=True, force_ascii=False)

(OUTPUT_DIR / "prompt_config.json").write_text(json.dumps(PROMPT_CONFIG, indent=2, ensure_ascii=False))
(OUTPUT_DIR / "dataset_report.json").write_text(json.dumps(report, indent=2, ensure_ascii=False))

for p in sorted(OUTPUT_DIR.iterdir()):
    print(f"{p.name:28} {p.stat().st_size / 1024:8.1f} KB")

# Round-trip check
_reloaded = pd.read_parquet(OUTPUT_DIR / "train.parquet")
assert len(_reloaded) == len(train_df) and isinstance(_reloaded.iloc[0]["messages"][0]["content"], str)
print("✅ Parquet round-trip OK")

## 14. Push to Hugging Face Hub
Creates `<your-username>/enterprise-pii-ticket-sft` with `train` / `validation` / `test` splits, uploads `prompt_config.json` + `dataset_report.json`, and appends a dataset card while preserving the auto-generated YAML metadata.

In [ ]:
if PUSH_TO_HUB:
    from datasets import Dataset, DatasetDict
    from huggingface_hub import DatasetCard, HfApi

    hf_splits = DatasetDict({
        name: Dataset.from_pandas(df[EXPORT_COLS].reset_index(drop=True), preserve_index=False)
        for name, df in splits.items()
    })
    hf_splits.push_to_hub(HF_DATASET_REPO, private=HF_PRIVATE, token=HF_TOKEN,
                          commit_message="Add processed SFT splits")

    api = HfApi(token=HF_TOKEN)
    for fname in ["prompt_config.json", "dataset_report.json"]:
        api.upload_file(path_or_fileobj=str(OUTPUT_DIR / fname), path_in_repo=fname,
                        repo_id=HF_DATASET_REPO, repo_type="dataset", commit_message=f"Add {fname}")

    card = DatasetCard.load(HF_DATASET_REPO, token=HF_TOKEN)
    card.text = f"""
# Enterprise PII Masking & Ticket Routing — SFT dataset

Instruction-tuning data (ChatML `messages`) for fine-tuning a small language model to return a strict JSON object:
`masked_text`, `detected_pii_types`, `ticket_category`, `urgency_level`.

* **Source:** [`{SOURCE_DATASET}`](https://huggingface.co/datasets/{SOURCE_DATASET}) (English subset) + rule-based synthetic enterprise tickets.
* **PII tags:** {", ".join(f"`[{t}]`" for t in PII_TAGS)}
* **Departments:** {", ".join(f"`{c}`" for c in CATEGORIES)}
* **Splits:** {", ".join(f"{k}={len(v):,}" for k, v in splits.items())}

## Label provenance
* PII spans come from the source dataset's character offsets (validated and repaired), or from synthetic generation (gold).
* `ticket_category` and `urgency_level` on real rows are **weak labels** from a transparent keyword lexicon with an ambiguity filter; synthetic rows carry gold labels (`origin` column).
* Synthetic credit cards are Luhn-valid test-range numbers, phone numbers use the reserved 555-01XX range, e-mail domains are fictitious, API keys use non-provider prefixes.

## Stats
```json
{json.dumps(report, indent=2)}
```
"""
    card.push_to_hub(HF_DATASET_REPO, token=HF_TOKEN, repo_type="dataset")
    print(f"✅ Dataset live at https://huggingface.co/datasets/{HF_DATASET_REPO}")
else:
    print("Skipping Hub push (local-only mode). Files are in", OUTPUT_DIR.resolve())

## ✅ Done
Notebook **02** loads `HF_DATASET_REPO` (or the local Parquet files) plus `prompt_config.json`, so the system prompt and schema used during training are byte-identical to the ones used here.